# Spark MLlib: classifying social-media texts into six emotions to flag distress

In [0]:
# %pip install seaborn

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (RegexTokenizer, StopWordsRemover, NGram, CountVectorizer, IDF,
                                VectorAssembler, IndexToString)
from pyspark.ml.classification import LogisticRegression, NaiveBayes, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder

import GX_utils as U

spark = U.get_spark("GX_MLlib")
U.setup_storage(spark)
silver = U.read_table(spark, "silver", "mental_health")

In [0]:
train_raw, test_raw = silver.randomSplit([0.8, 0.2], seed=U.SEED)

class_counts = {r["label"]: r["count"] for r in train_raw.groupBy("label").count().collect()}
n_train = sum(class_counts.values())
class_weights = {label: n_train / (len(class_counts) * count) for label, count in class_counts.items()}
weight_map = F.create_map(*[F.lit(x) for kv in class_weights.items() for x in kv])

train_df = U.maybe_cache(train_raw.withColumn("class_weight", weight_map[F.col("label")]))
test_df = U.maybe_cache(test_raw)
U.write_table(test_df.select("ID"), "gold", "ml_test_ids")

print(f"train: {train_df.count():,} | test: {test_df.count():,}")
pd.DataFrame({"emotion": U.EMOTION_LABELS,
              "train_texts": [class_counts[float(i)] for i in range(6)],
              "weight": [round(class_weights[float(i)], 2) for i in range(6)]})

In [0]:
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
distress_idx = [U.EMOTION_LABELS.index(e) for e in U.DISTRESS_EMOTIONS]


def confusion(predictions):
    counts = predictions.groupBy("label", "prediction").count().toPandas()
    labels = [float(i) for i in range(len(U.EMOTION_LABELS))]
    return (counts.pivot_table(index="label", columns="prediction", values="count", fill_value=0)
                  .reindex(index=labels, columns=labels, fill_value=0))


def class_report(cm):
    m = cm.values.astype(float)
    tp = np.diag(m)
    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.nan_to_num(tp / m.sum(axis=0))
        recall = np.nan_to_num(tp / m.sum(axis=1))
        f1 = np.nan_to_num(2 * precision * recall / (precision + recall))
    return pd.DataFrame({"emotion": U.EMOTION_LABELS, "precision": precision, "recall": recall,
                         "f1": f1, "support": m.sum(axis=1).astype(int)}).round(3)


def summarize(name, predictions):
    cm = confusion(predictions)
    m = cm.values.astype(float)
    report = class_report(cm)
    return {"model": name,
            "accuracy": float(np.trace(m) / m.sum()),
            "macro_f1": float(report["f1"].mean()),
            "weighted_f1": float(f1_evaluator.evaluate(predictions)),
            "distress_recall": float(m[np.ix_(distress_idx, distress_idx)].sum() / m[distress_idx, :].sum())}

In [0]:
style_pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=["word_count", "char_length", "upper_ratio"], outputCol="style_features"),
    RandomForestClassifier(featuresCol="style_features", labelCol="label", numTrees=50, maxDepth=8, seed=U.SEED),
])
style_predictions = style_pipeline.fit(train_df).transform(test_df)
style_summary = summarize("Style only (length + capitals)", style_predictions)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in style_summary.items()})
class_report(confusion(style_predictions))

In [0]:
ml_stopwords = sorted({w.replace("'", "") for w in StopWordsRemover.loadDefaultStopWords("english")} - U.NEGATIONS)

feature_stages = [
    RegexTokenizer(inputCol="cleaned_text", outputCol="tokens", pattern="\\s+", minTokenLength=2),
    StopWordsRemover(inputCol="tokens", outputCol="terms", stopWords=ml_stopwords),
    NGram(n=2, inputCol="terms", outputCol="bigrams"),
    CountVectorizer(inputCol="terms", outputCol="unigram_tf", vocabSize=20000, minDF=3),
    CountVectorizer(inputCol="bigrams", outputCol="bigram_tf", vocabSize=10000, minDF=5),
    IDF(inputCol="unigram_tf", outputCol="unigram_tfidf"),
    IDF(inputCol="bigram_tf", outputCol="bigram_tfidf"),
    VectorAssembler(inputCols=["unigram_tfidf", "bigram_tfidf"], outputCol="features"),
]

feature_model = Pipeline(stages=feature_stages).fit(train_df)
train_features = U.maybe_cache(feature_model.transform(train_df).select("label", "class_weight", "features"))
fit_part, validation_part = train_features.randomSplit([0.85, 0.15], seed=U.SEED)
print(f"unigrams: {len(feature_model.stages[3].vocabulary):,} | bigrams: {len(feature_model.stages[4].vocabulary):,}")

In [0]:
candidates = {
    "Logistic Regression": LogisticRegression(labelCol="label", featuresCol="features", weightCol="class_weight",
                                              maxIter=50, regParam=0.02, family="multinomial"),
    "Naive Bayes": NaiveBayes(labelCol="label", featuresCol="features", weightCol="class_weight",
                              modelType="multinomial", smoothing=1.0),
}

comparison = [summarize(name, est.fit(fit_part).transform(validation_part)) for name, est in candidates.items()]
pd.DataFrame(comparison).round(3).sort_values("macro_f1", ascending=False)

In [0]:
lr = LogisticRegression(labelCol="label", featuresCol="features", weightCol="class_weight",
                        maxIter=50, family="multinomial")
param_grid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.005, 0.02, 0.08])
    .addGrid(lr.elasticNetParam, [0.0, 0.3])
    .build())

tuning_results = pd.DataFrame([
    {**{p.name: v for p, v in params.items()},
     "validation_weighted_f1": f1_evaluator.evaluate(lr.copy(params).fit(fit_part).transform(validation_part))}
    for params in param_grid
]).sort_values("validation_weighted_f1", ascending=False)

best_params = {k: float(tuning_results.iloc[0][k]) for k in ["regParam", "elasticNetParam"]}
print("best:", best_params)
tuning_results.round(4)

In [0]:
final_pipeline = Pipeline(stages=feature_stages + [
    LogisticRegression(labelCol="label", featuresCol="features", weightCol="class_weight",
                       maxIter=100, family="multinomial", **best_params),
    IndexToString(inputCol="prediction", outputCol="predicted_emotion", labels=U.EMOTION_LABELS),
])
final_model = final_pipeline.fit(train_df)
test_predictions = U.maybe_cache(final_model.transform(test_df))

final_summary = summarize("Tuned Logistic Regression (test)", test_predictions)
test_cm = confusion(test_predictions)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in final_summary.items()})
class_report(test_cm)

In [0]:
row_share = test_cm.div(test_cm.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(row_share, annot=True, fmt=".2f", cmap="mako_r", cbar=False,
            xticklabels=U.EMOTION_LABELS, yticklabels=U.EMOTION_LABELS, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix (share of each actual class)", fontweight="bold")
plt.tight_layout(); plt.show()

In [0]:
(test_predictions
    .withColumn("length_group", F.when(F.col("word_count") <= 60, "short (<=60 words)").otherwise("long (>60 words)"))
    .groupBy("Emotion", "length_group")
    .agg(F.count("*").alias("texts"),
         F.round(F.avg((F.col("prediction") == F.col("label")).cast("int")), 3).alias("accuracy"))
    .orderBy("Emotion", "length_group")
    .show())

short_summary = summarize("Tuned Logistic Regression (test, <=60 words)", test_predictions.filter("word_count <= 60"))
print({k: round(v, 3) if isinstance(v, float) else v for k, v in short_summary.items()})

In [0]:
lr_model = final_model.stages[-2]
vocabulary = final_model.stages[3].vocabulary + final_model.stages[4].vocabulary
coefficients = lr_model.coefficientMatrix.toArray()

pd.DataFrame({emotion: [vocabulary[i] for i in np.argsort(-coefficients[k])[:12]]
              for k, emotion in enumerate(U.EMOTION_LABELS)})

In [0]:
model_path = U.work_path("models", "emotion_classifier")
final_model.write().overwrite().save(model_path)
print("model saved to", model_path)

all_results = [style_summary] + comparison + [final_summary, short_summary]
U.write_table(spark.createDataFrame(pd.DataFrame(all_results)), "gold", "model_metrics")
pd.DataFrame(all_results).round(3)

In [0]:
print(f"Distress recall on the test set: {final_summary['distress_recall']:.1%} "
      f"(macro F1 {final_summary['macro_f1']:.3f}); style-only baseline: {style_summary['macro_f1']:.3f} macro F1; "
      f"short texts only: {short_summary['macro_f1']:.3f} macro F1.")